# 03 - 函数（Java vs Python）

只讲你在 AI 应用开发里最常用、且和 Java 差异明显的部分。


## 今日目标（25-35 分钟）

- 掌握 `def`、默认参数、关键字参数
- 会用 `*args / **kwargs` 写可扩展函数
- 看懂并写出最基础的 `lambda`、装饰器
- 理解“函数是一等公民”在 Python 里的实际用法

完成标准：通过末尾打卡题，独立写出一个带装饰器的函数。


## 1. `def` + 类型注解（Type Hints）


In [ ]:
# 说明：本段示例对应「1. def + 类型注解（Type Hints）」，建议先看注释再运行，重点观察输出与预期是否一致。

# Java: String greet(String name, int age) { ... }
def greet(name: str, age: int) -> str:
    return f"{name} is {age}"

print(greet("kai", 18))
print(greet(123, "abc"))  # 注解默认不做运行时强校验


In [ ]:
# 说明：演示主动抛出异常，尽早拦截非法输入，避免问题向后传播。

def split_name(full_name: str) -> tuple[str, str]:
    parts = full_name.split()
    # 这里显式校验输入格式，避免新手把 IndexError 误解成函数语法问题
    if len(parts) != 2:
        raise ValueError("full_name 必须是'名 空格 姓'格式")
    return parts[0], parts[1]

first, last = split_name("Kai Li")
print(first, last)


## 2. 默认参数 + 关键字参数


In [ ]:
# 说明：本段示例对应「2. 默认参数 + 关键字参数」，建议先看注释再运行，重点观察输出与预期是否一致。

def create_user(name: str, role: str = "user", active: bool = True) -> dict:
    return {"name": name, "role": role, "active": active}

print(create_user("kai"))
print(create_user("kai", "admin"))
print(create_user(name="kai", active=False))


In [ ]:
# 说明：本段示例对应「2. 默认参数 + 关键字参数」，建议先看注释再运行，重点观察输出与预期是否一致。

def send_message(content: str, *, channel: str = "email") -> str:
    return f"send via {channel}: {content}"

# * 后面的参数只能用关键字传，能减少调用方误传位置参数
print(send_message("hello", channel="sms"))

# 💡 实战：FastAPI 的路由函数参数本质上也是强制关键字参数
# @app.get("/users")
# async def get_users(page: int = 1, size: int = 10):  ← 调用方必须显式传参名
# 理解 * 分隔有助于读懂 FastAPI 和各种库的函数签名


In [ ]:
# 说明：本段示例对应「2. 默认参数 + 关键字参数」，建议先看注释再运行，重点观察输出与预期是否一致。

# 经典坑：可变对象不要当默认参数

def append_item_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

print(append_item_bad("a"))
print(append_item_bad("b"))  # 会复用上一次的列表

# ⚠️ 坑：这是 Python 面试必考题，也是真实 bug 来源
# 根本原因：默认参数在"函数定义时"只创建一次，不是每次调用都创建
# [] 这个列表对象在函数定义时就固定了，所有调用共享它
# 只要记住：默认参数是可变对象（list/dict/set）时，一律用 None 替代

def append_item_good(item, bucket=None):
    # 用 None 作为哨兵值，避免跨调用共享同一份可变对象
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(append_item_good("a"))
print(append_item_good("b"))


## 3. `*args / **kwargs`


In [ ]:
# 说明：本段示例对应「3. *args / **kwargs」，建议先看注释再运行，重点观察输出与预期是否一致。

# Java: int sum(int... nums)
def sum_all(*nums: int) -> int:
    total = 0
    for n in nums:
        total += n
    return total

print(sum_all(1, 2, 3))
print(sum_all(10, 20, 30, 40))


In [ ]:
# 说明：演示 **kwargs 收集可选命名参数，便于函数按需扩展字段。

def build_profile(name: str, **kwargs) -> dict:
    profile = {"name": name}
    profile.update(kwargs)
    return profile

print(build_profile("kai", city="shanghai", skill="python"))


In [ ]:
# 说明：演示函数同时接收可变位置参数和关键字参数，提升可扩展性。

def api_call(path: str, method: str = "GET", *tags: str, **options):
    print("path =", path)
    print("method =", method)
    print("tags =", tags)
    print("options =", options)

api_call("/users", "POST", "beta", "internal", timeout=10, retry=2)

# ⚠️ 坑：混合使用时参数顺序必须严格遵守：
# 普通参数 → 默认参数 → *args → 关键字专用参数 → **kwargs
# 顺序写错会直接 SyntaxError，Python 不会帮你猜意图


## 4. `lambda`（匿名函数）


In [ ]:
# 说明：演示 lambda 匿名函数在排序、过滤等一次性小逻辑中的用法。

users = [
    {"name": "a", "score": 88},
    {"name": "b", "score": 95},
    {"name": "c", "score": 91},
]

# lambda 适合“只用一次的短逻辑”
sorted_users = sorted(users, key=lambda u: u["score"], reverse=True)
print(sorted_users)


In [ ]:
# 说明：演示 lambda 匿名函数在排序、过滤等一次性小逻辑中的用法。

nums = [1, 2, 3, 4, 5, 6]

# filter/map 写法（Java 开发者熟悉，因为 Stream API）
even_nums = list(filter(lambda x: x % 2 == 0, nums))
squared = list(map(lambda x: x * x, nums))

print(even_nums)
print(squared)

# [Review] 但 Python 社区更推荐用列表推导式（07 会详细讲），更简洁易读：
# even_nums = [x for x in nums if x % 2 == 0]
# squared = [x * x for x in nums]
# 实际开发中优先用列表推导式，filter/map 知道能看懂就行


## 5. 函数是一等公民


In [ ]:
# 说明：本段示例对应「5. 函数是一等公民」，建议先看注释再运行，重点观察输出与预期是否一致。

def to_upper(text: str) -> str:
    return text.upper()

handler = to_upper
print(handler("hello"))


In [ ]:
# 说明：本段示例对应「5. 函数是一等公民」，建议先看注释再运行，重点观察输出与预期是否一致。

def run_twice(func, value):
    return func(func(value))


def add_exclaim(text: str) -> str:
    return text + "!"

print(run_twice(add_exclaim, "hi"))


## 6. 装饰器（Decorator）

FastAPI 里的 `@app.get()` 本质也是装饰器。


In [ ]:
# 说明：演示无参装饰器如何在函数执行前后插入日志逻辑。

from functools import wraps


def log_call(func):
    # [Review] @wraps(func) 的作用：保留被装饰函数的原始名称和文档
    # 不加的话，fetch_user.__name__ 会变成 "wrapper" 而不是 "fetch_user"
    # 调试和日志时会很困惑，所以写装饰器时养成习惯加上
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[start] {func.__name__}")
        result = func(*args, **kwargs)
        print(f"[end] {func.__name__}")
        return result

    return wrapper


@log_call
def fetch_user(user_id: int) -> dict:
    return {"id": user_id, "name": "kai"}

print(fetch_user(1001))
print(f"函数名: {fetch_user.__name__}")  # 因为 @wraps，这里是 "fetch_user" 而不是 "wrapper"

# 💡 实战：不加 @wraps 会导致 FastAPI 把所有路由识别为同名的 "wrapper" 函数，
# 造成路由冲突或 OpenAPI 文档混乱，是真实项目里出现过的 bug


In [ ]:
# 说明：演示带参数装饰器的三层结构（参数层/装饰层/执行层）。

# [Review] 带参数的装饰器：三层嵌套，外层接收装饰器参数，中层接收函数，内层接收函数参数
from functools import wraps


def route(path: str):
    def decorator(func):
        @wraps(func)  # [Review] 补上 @wraps，和上面保持一致
        def wrapper(*args, **kwargs):
            print(f"routing: {path}")
            return func(*args, **kwargs)

        return wrapper

    return decorator


@route("/health")
def health_check():
    return "ok"

print(health_check())


## 7. Java 对照速记

| Java | Python |
|---|---|
| `String f(String x)` | `def f(x):` |
| `int sum(int... nums)` | `def sum(*nums):` |
| `Map<String, Object>` 参数扩展 | `**kwargs` |
| `x -> x.getScore()` | `lambda x: x["score"]` |
| 注解/拦截器 | `@decorator` |
| 函数式接口传参 | 函数直接作为参数传递 |


## 今日打卡题

给定：

```python
students = [
    {"name": "alice", "score": 81},
    {"name": "bob", "score": 95},
    {"name": "cindy", "score": 88},
]
```

请完成：
1. 写函数 `top_student(data)`，返回分数最高的学生字典
2. 写函数 `build_tags(*tags, **meta)`，返回 `{"tags": tags, "meta": meta}`
3. 写一个装饰器 `trace`，调用函数前后打印 `enter/exit`


In [ ]:
# 说明：这是练习留空区域，建议先独立完成，再运行后续参考答案进行对照。

students = [
    {"name": "alice", "score": 81},
    {"name": "bob", "score": 95},
    {"name": "cindy", "score": 88},
]

# TODO: 先自己写


In [ ]:
# 说明：这是打卡题参考实现，综合了 lambda、*args/**kwargs 与装饰器。

students = [
    {"name": "alice", "score": 81},
    {"name": "bob", "score": 95},
    {"name": "cindy", "score": 88},
]


def top_student(data):
    return max(data, key=lambda s: s["score"])


def build_tags(*tags, **meta):
    return {"tags": tags, "meta": meta}


def trace(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("enter")
        result = func(*args, **kwargs)
        print("exit")
        return result

    return wrapper


@trace
def hello(name):
    print(f"hello, {name}")


print(top_student(students))
print(build_tags("python", "fastapi", level="junior", days=10))
hello("kai")


下一步建议：进入 `04-class-oop.ipynb`，重点理解 `self`、`dataclass`、继承差异。
